# Behavioral Drift Features
Behavioral drift features that compare a customer’s current behavior with previous behavior.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_parquet("../data/standardized.parquet")

# Create a completed-purchases view for transaction analysis; keep the full data unchanged.
purchases = df.loc[
    ~df["Is_Cancellation"] & df["Quantity"].gt(0)
].copy()
analysis_cols = ["Quantity", "Price", "Total_Price"]

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (776591, 24)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Is_Cancellation', 'Total_Price', 'CustomerMonth', 'Country_grouped', 'Country_EIRE', 'Country_France', 'Country_Germany', 'Country_Netherlands', 'Country_Other', 'Country_United Kingdom', 'Quantity_log', 'Price_log', 'Total_Price_log', 'Quantity_scaled', 'Price_scaled', 'Total_Price_scaled']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Is_Cancellation,Total_Price,...,Country_Germany,Country_Netherlands,Country_Other,Country_United Kingdom,Quantity_log,Price_log,Total_Price_log,Quantity_scaled,Price_scaled,Total_Price_scaled
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,83.4,...,False,False,False,True,2.564949,2.073172,4.435567,0.661217,1.585922,2.002152
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,...,False,False,False,True,2.564949,2.047693,4.406719,0.661217,1.541560,1.972839
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,...,False,False,False,True,2.564949,2.047693,4.406719,0.661217,1.541560,1.972839
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,100.8,...,False,False,False,True,3.891820,1.131402,4.623010,2.010135,-0.053821,2.192616
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,30.0,...,False,False,False,True,3.218876,0.810930,3.433987,1.326009,-0.611804,0.984430


In [2]:
CHURN_WINDOW_DAYS = 90

#one row per customer from the dataframe
df = df.dropna(subset=["Customer ID"]).copy()
df["Customer ID"] = df["Customer ID"].astype(int) #change object to integer
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"]) #convert to datetime format
df["Revenue"] = df["Total_Price"]                    # revenue per line item
df["is_cancel"] = df["Is_Cancellation"].astype(bool)  # cancelled/returned lines
 
# Features come only from before the cutoff of 90 days,  churn label comes from after it
cutoff = df["InvoiceDate"].max() - pd.Timedelta(days=CHURN_WINDOW_DAYS)
obs = df[df["InvoiceDate"] <= cutoff] # observations before the cutoff
obs_p = obs[~obs["is_cancel"]] # observations before the cutoff that are NOT cancelled
future_ids = df.loc[(df["InvoiceDate"] > cutoff) & ~df["is_cancel"], "Customer ID"].unique()
 ## ^customers who made purchases after the cutoff and were not cancelled 

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Is_Cancellation,Total_Price,...,Country_Other,Country_United Kingdom,Quantity_log,Price_log,Total_Price_log,Quantity_scaled,Price_scaled,Total_Price_scaled,Revenue,is_cancel
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,83.4,...,False,True,2.564949,2.073172,4.435567,0.661217,1.585922,2.002152,83.4,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,...,False,True,2.564949,2.047693,4.406719,0.661217,1.541560,1.972839,81.0,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,81.0,...,False,True,2.564949,2.047693,4.406719,0.661217,1.541560,1.972839,81.0,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,100.8,...,False,True,3.891820,1.131402,4.623010,2.010135,-0.053821,2.192616,100.8,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,30.0,...,False,True,3.218876,0.810930,3.433987,1.326009,-0.611804,0.984430,30.0,False


## Purchase frequency

In [3]:
# Two adjacent windows, both entirely before the cutoff (no leakage into the churn period)
recent_start = cutoff - pd.Timedelta(days=CHURN_WINDOW_DAYS)
prior_start  = recent_start - pd.Timedelta(days=CHURN_WINDOW_DAYS)

recent = obs_p[obs_p["InvoiceDate"] > recent_start]
prior  = obs_p[(obs_p["InvoiceDate"] > prior_start) & (obs_p["InvoiceDate"] <= recent_start)]

# Purchase frequency = number of distinct invoices per customer in each window
# NOTE: use nunique on the invoice column because each invoice has many line items
freq_recent = recent.groupby("Customer ID")["Invoice"].nunique().rename("freq_recent")
freq_prior  = prior.groupby("Customer ID")["Invoice"].nunique().rename("freq_prior")

# One row per customer seen before the cutoff; no purchases in a window -> 0
features = pd.DataFrame(index=pd.Index(obs_p["Customer ID"].unique(), name="Customer ID"))
features = features.join(freq_recent).join(freq_prior).fillna(0)

# Drift features
features["freq_diff"]  = features["freq_recent"] - features["freq_prior"]
features["freq_ratio"] = (features["freq_recent"] + 1) / (features["freq_prior"] + 1)
features["freq_pct_change"] = np.where(
    features["freq_prior"] > 0,
    features["freq_diff"] / features["freq_prior"],
    np.nan,
)

# Flag customers whose first-ever purchase is inside the recent window, since their
# "prior" frequency of 0 = "not a customer yet", not "dropped activity"
first_purchase = obs_p.groupby("Customer ID")["InvoiceDate"].min()
features["is_new_in_recent"] = (first_purchase.reindex(features.index) > recent_start).astype(int)

# Churn label: 1 if the customer made no purchase after the cutoff
features["churn"] = (~features.index.isin(future_ids)).astype(int)

features.head()

,freq_recent,freq_prior,freq_diff,freq_ratio,freq_pct_change,is_new_in_recent,churn
Customer ID,,,,,,,
13085,1.0,0.0,1.0,2.000000,NaN,0,1
13078,5.0,4.0,1.0,1.200000,0.25,0,0
15362,0.0,0.0,0.0,1.000000,NaN,0,1
18102,10.0,10.0,0.0,1.000000,0.00,0,0
12682,9.0,5.0,4.0,1.666667,0.80,0,0


- **freq_recent**: The number of distinct invoices the customer placed in the 90 days just before the cutoff.
- **freq_prior**: The number of distinct invoices the customer placed in the 90 days before that recent window.
- **freq_diff**: The raw change in purchase count (recent minus prior), where negative values mean the customer is buying less often.
- **freq_ratio**: The recent frequency divided by the prior frequency, with 1 added to each so customers with no prior purchases don't cause a divide-by-zero, where values below 1 signal slowing down.
- **freq_pct_change**: The change in purchase count as a fraction of the prior count (e.g., -0.5 means purchases dropped by half), left as NaN when the customer had no prior purchases.
- **is_new_in_recent**: A 1/0 flag marking customers whose first-ever purchase falls in the recent window, so a prior frequency of 0 isn't mistaken for a drop in activity.

## Revenue

In [4]:
# Revenue per customer in each window (cancellations already excluded via obs_p)
rev_recent = recent.groupby("Customer ID")["Revenue"].sum().rename("rev_recent")
rev_prior  = prior.groupby("Customer ID")["Revenue"].sum().rename("rev_prior")

features = features.join(rev_recent).join(rev_prior)
features[["rev_recent", "rev_prior"]] = features[["rev_recent", "rev_prior"]].fillna(0)

# Drift features
features["rev_diff"] = features["rev_recent"] - features["rev_prior"]

# Log ratio: log(1 + recent) - log(1 + prior). Handles zeros, and damps the heavy
# right skew in spend so big spenders don't dominate.
features["rev_log_ratio"] = np.log1p(features["rev_recent"]) - np.log1p(features["rev_prior"])

features["rev_pct_change"] = np.where(
    features["rev_prior"] > 0,
    features["rev_diff"] / features["rev_prior"],
    np.nan,  # undefined when there was no prior spend
)

features[["rev_recent", "rev_prior", "rev_diff", "rev_log_ratio", "rev_pct_change"]].head()

,rev_recent,rev_prior,rev_diff,rev_log_ratio,rev_pct_change
Customer ID,,,,,
13085,137.98,0.00,137.98,4.934330,NaN
13078,2426.79,1626.40,800.39,0.399998,0.492124
15362,0.00,0.00,0.00,0.000000,NaN
18102,28349.76,36435.50,-8085.74,-0.250917,-0.221919
12682,2654.63,1669.75,984.88,0.463409,0.589837


- **Log ratio:** the data is symmetric where 0 means no change, negative means spending decreased.
- **rev_recent:** The customer’s total revenue (excluding cancellations) in the 90 days just before the cutoff.
- **rev_prior:** The customer’s total revenue in the 90 days before that recent window.
- **rev_diff:** The raw change in spend (recent minus prior), where negative values mean the customer is spending less than before.
- **rev_log_ratio:** The difference between the log of recent and prior revenue (each plus 1), which measures proportional change so that 0 means no change, negative means spending fell, and large spenders don’t dominate.
- **rev_pct_change:** The change in spend as a fraction of prior revenue (e.g., -0.5 means spend dropped by half), left as NaN when the customer had no prior spend.

## Unique Items

In [5]:
items_recent = recent.groupby("Customer ID")["StockCode"].nunique().rename("items_recent")
items_prior  = prior.groupby("Customer ID")["StockCode"].nunique().rename("items_prior")

features = features.join(items_recent).join(items_prior)
features[["items_recent", "items_prior"]] = features[["items_recent", "items_prior"]].fillna(0)

# Drift features
features["items_diff"]  = features["items_recent"] - features["items_prior"]
features["items_ratio"] = (features["items_recent"] + 1) / (features["items_prior"] + 1)  # smoothed
features["items_pct_change"] = np.where(
    features["items_prior"] > 0,
    features["items_diff"] / features["items_prior"],
    np.nan,  # undefined when there were no prior items
)

# Share of recent items the customer had never bought in the prior window (variety-seeking vs. repeat buying)
recent_pairs = recent[["Customer ID", "StockCode"]].drop_duplicates()
prior_pairs  = prior[["Customer ID", "StockCode"]].drop_duplicates()
merged = recent_pairs.merge(prior_pairs, on=["Customer ID", "StockCode"], how="left", indicator=True)
new_items = (merged["_merge"] == "left_only").groupby(merged["Customer ID"]).sum().rename("items_new_recent")

features = features.join(new_items)
features["items_new_recent"] = features["items_new_recent"].fillna(0)
features["items_new_share"] = np.where(
    features["items_recent"] > 0,
    features["items_new_recent"] / features["items_recent"],
    np.nan,  # undefined when there were no recent purchases
)

features[["items_recent", "items_prior", "items_diff", "items_ratio",
          "items_pct_change", "items_new_recent", "items_new_share"]].head()

,items_recent,items_prior,items_diff,items_ratio,items_pct_change,items_new_recent,items_new_share
Customer ID,,,,,,,
13085,9.0,0.0,9.0,10.000000,NaN,9.0,1.000000
13078,52.0,49.0,3.0,1.060000,0.061224,20.0,0.384615
15362,0.0,0.0,0.0,1.000000,NaN,0.0,NaN
18102,17.0,47.0,-30.0,0.375000,-0.638298,1.0,0.058824
12682,78.0,67.0,11.0,1.161765,0.164179,44.0,0.564103


- **items_recent**: The number of distinct products (by StockCode) the customer bought in the 90 days just before the cutoff.
- **items_prior**: The number of distinct products the customer bought in the 90 days before that recent window.
- **items_diff**: The raw change in distinct product count (recent minus prior), where negative values = customer buying a narrower range of products.
- **items_ratio**: The recent distinct item count divided by the prior count, with 1 added to each to avoid divide-by-zero, where values below 1 signal a small product variety.
- **items_pct_change**: The change in distinct item count as a fraction of the prior count (ex. -0.5 means variety was cut in half), left as NaN when the customer had no prior items.
- **items_new_recent**: The number of distinct products the customer bought in the recent window that they had not bought in the prior window.
- **items_new_share**: The fraction of the customer's recent distinct products that were new compared to the prior window, where higher values suggest exploring and lower values suggest repeat buying, left as NaN when there were no recent purchases.